# OCT classification: Basic CNN, Dynamic CNN and noise-aware training

用同一训练集比较四种方法，观察跨数据集泛化。**先试跑，再完整训练。**

- 选择 `Runtime → Change runtime type → GPU`。
- 准备小组当前清洗后的 OCT2017 ZIP；完整模式另外需要 NEH 与 OCT-C8 ZIP。
- 数据只在自己的 Colab / Drive 中读取，不上传到 GitHub。
- 当前每种方法运行一次；固定 seed42 不代表多随机种子实验。
- 本 notebook 不带预先填入的实验结果。短程试跑不是论文结果。

完整说明见 [README](https://github.com/jiangyucheng2006-code/oct-cross-dataset-generalization)。

## 1. 获取固定版本的代码并安装依赖

使用标签 `v0.1.0` 保持恢复训练时的代码一致。不要在同一 run 中切换版本。
Colab 已提供 PyTorch；下面不会更换它的 CUDA 安装。

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/jiangyucheng2006-code/oct-cross-dataset-generalization.git"
REPO_REF = "v0.1.0"
REPO = Path("/content/oct-cross-dataset-generalization")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONHASHSEED"] = "42"
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", REPO_REF, "--depth", "1", REPO_URL, str(REPO)], check=True)
commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
expected_commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", REPO_REF], text=True).strip()
assert commit == expected_commit, "Existing checkout differs from REPO_REF; use a new runtime or restore that version."
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
print("Code commit:", commit)

## 2. 挂载 Drive 和填写参数

修改下面三个 ZIP 路径。共享文件夹可以先添加到自己 My Drive，再填快捷方式路径。
文件名可以不同；完整模式会检查内容 SHA256。

`MODE="smoke"`：每类最多64张训练和64张验证图，两轮，不使用测试图。
`MODE="full"`：完整训练，上限100轮，validation early stopping。

可选方法：`basic`, `dynamic`, `basic_noise`, `na_dynamic`。先用 `basic` 试跑。
完整四组比较时将 `VARIANTS` 改为这四项；开始后不要按外部成绩改名单。
变更模型列表、配置或数据时换一个新的 `RUN_NAME`。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
MODE = "smoke"  # "smoke" or "full"
VARIANTS = ["basic"]  # full comparison: ["basic", "dynamic", "basic_noise", "na_dynamic"]
RUN_NAME = "trial_01"  # same name to resume; new name for changed settings

OCT2017_ZIP = Path("/content/drive/MyDrive/OCT_project_data/OCT2017-cleaned.zip")
NEH_ZIP = Path("/content/drive/MyDrive/OCT_project_data/NEH-cleaned.zip")
OCT_C8_ZIP = Path("/content/drive/MyDrive/OCT_project_data/OCT-C8-cleaned.zip")
DRIVE_RESULTS = Path("/content/drive/MyDrive/OCT_project_results")

# Start with the matched server recipe. Reducing batch size changes the experiment
# (including BatchNorm); keep the same value across all compared methods.
BATCH_SIZE = 64
MAX_EPOCHS = 100
RUN_FINAL_EVALUATION = False  # set True only AFTER complete full-data training

import json
import re
import shutil
import torch

assert MODE in ("smoke", "full")
assert re.fullmatch(r"[A-Za-z0-9_-]+", RUN_NAME), "Use letters, numbers, _ and - in RUN_NAME."
assert VARIANTS and len(VARIANTS) == len(set(VARIANTS))
assert set(VARIANTS) <= {"basic", "dynamic", "basic_noise", "na_dynamic"}
assert torch.cuda.is_available(), "Enable a GPU runtime before training."
print("GPU:", torch.cuda.get_device_name(), "| PyTorch:", torch.__version__)
AUDIT = Path("/content/oct_data") / ("audit_" + MODE)
RUN_ROOT = Path("/content/oct_runs") / MODE / RUN_NAME
MIRROR = DRIVE_RESULTS / MODE / RUN_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=True)
MIRROR.mkdir(parents=True, exist_ok=True)
print("Free runtime disk GB:", round(shutil.disk_usage('/content').free / 1e9, 1))
if MODE == "full":
    assert shutil.disk_usage('/content').free > 12e9, "Full caches need roughly 8 GB; allow 12 GB free."

## 3. 读取现有划分并准备缓存

保留 ZIP 中 train/val/test 的分配，不重新分数据。full 模式先检查三份 ZIP
的校验值，再做 filename patient-ID 和 exact-content 检查。预计需要数分钟或更久。
完整模式先将 ZIP 复制到 `/content`，减少 Drive 随机读取，需约25GB可用空间。
图像缓存也写到 `/content`，原始 ZIP 不变。如果输入是 Drive 外层 ZIP，下面只解出
其中唯一一个内层 ZIP；不会将图像解压到 Drive。

In [ ]:
import zipfile
import zlib

def unwrap_single_zip(path):
    if not path.is_file():
        raise FileNotFoundError(f"Check the Drive path: {path}")
    with zipfile.ZipFile(path) as archive:
        names = [info for info in archive.infolist() if not info.is_dir()]
        image_suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
        if any(Path(info.filename).suffix.lower() in image_suffixes for info in names):
            return path
        nested = [info for info in names if info.filename.lower().endswith(".zip")]
        if len(nested) != 1:
            raise ValueError("Expected images or exactly one nested dataset ZIP: " + str(path))
        target = Path("/content/oct_archives") / Path(nested[0].filename).name
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists() and target.stat().st_size == nested[0].file_size:
            crc = 0
            with target.open("rb") as handle:
                for chunk in iter(lambda: handle.read(4 * 1024 * 1024), b""):
                    crc = zlib.crc32(chunk, crc)
            if crc == nested[0].CRC:
                return target
        with archive.open(nested[0]) as source, target.open("wb") as destination:
            shutil.copyfileobj(source, destination, 4 * 1024 * 1024)
        return target

def stage_for_full(path):
    if MODE != "full":
        return path
    target = Path("/content/oct_archives") / path.name
    target.parent.mkdir(parents=True, exist_ok=True)
    if path.resolve() == target.resolve():
        return path
    if target.exists() and target.stat().st_size == path.stat().st_size:
        return target  # full preparation verifies its SHA256 before use
    assert shutil.disk_usage('/content').free > path.stat().st_size + 12e9, "Need space for this ZIP and image caches."
    temporary = target.with_suffix(".copying")
    shutil.copyfile(path, temporary)
    temporary.replace(target)
    return target

source_zip = stage_for_full(unwrap_single_zip(OCT2017_ZIP))
args = [sys.executable, "-u", "src/prepare_data.py", "--mode", MODE,
        "--oct2017", str(source_zip), "--out", str(AUDIT)]
if MODE == "full":
    args += ["--neh", str(stage_for_full(unwrap_single_zip(NEH_ZIP))),
             "--oct-c8", str(stage_for_full(unwrap_single_zip(OCT_C8_ZIP)))]
subprocess.run(args, check=True)
audit = json.loads((AUDIT / "audit.json").read_text())
assert audit["passed"]
print(json.dumps(audit["retained_counts"], indent=2))

## 4. 保存本次实验配置

配置和方法名单在训练前保存到 Drive。正式实验只用 source validation 选 checkpoint。
固定 seed42 仅用于复现；这次不做多 seed 重复。不根据外部测试结果选择超参数。

In [ ]:
cfg = json.loads((REPO / "configs/config.json").read_text())
cfg.update(audit_dir=str(AUDIT), batch_size=BATCH_SIZE, workers=2, max_epochs=MAX_EPOCHS)
exp = json.loads((REPO / "configs/experiments.json").read_text())
exp.update(variants=VARIANTS, seeds=[42])
plan = {"mode": MODE, "variants": VARIANTS, "seed": 42, "commit": commit, "config": cfg}
plan_file = MIRROR / "plan.json"
if plan_file.exists():
    assert json.loads(plan_file.read_text()) == plan, "Plan changed: restore settings or choose a new RUN_NAME."
else:
    plan_file.write_text(json.dumps(plan, indent=2))
CONFIG = RUN_ROOT / "config.json"
EXPERIMENTS = RUN_ROOT / "experiments.json"
CONFIG.write_text(json.dumps(cfg, indent=2))
EXPERIMENTS.write_text(json.dumps(exp, indent=2))
shutil.copy2(CONFIG, MIRROR / "config.json")
shutil.copy2(EXPERIMENTS, MIRROR / "experiments.json")
print("Results/checkpoints:", MIRROR)

## 5. 训练或恢复

依次运行所选方法。每个完成的 epoch 都会保存 `last.pt`、最佳 `best.pt`、曲线和配置到 Drive。
如果 Colab 断开，重新执行上面各格，保持同一版本、路径、MODE 和 RUN_NAME，再执行这一格。
只加载自己可信运行生成的 checkpoint。`completed.json` 存在时跳过该已完成模型。

In [ ]:
for variant in VARIANTS:
    run = RUN_ROOT / f"{variant}_seed42"
    backup = MIRROR / run.name
    if not run.exists() and backup.exists():
        shutil.copytree(backup, run)
    if (run / "completed.json").exists():
        print("Already completed:", variant)
        continue
    command = [sys.executable, "-u", "src/train.py", "--config", str(CONFIG),
               "--variant", variant, "--seed", "42", "--out", str(run),
               "--archive", str(MIRROR), "--resume"]
    if MODE == "smoke":
        command += ["--smoke"]
    print("Training:", variant, "|", MODE, flush=True)
    subprocess.run(command, check=True)

## 6. 查看 source-validation 结果与训练曲线

这里只显示 validation。smoke 的数字仅用于检查代码能运行，不能用于比较方法效果。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rows = []
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for variant in VARIANTS:
    run = RUN_ROOT / f"{variant}_seed42"
    if not (run / "completed.json").exists():
        continue
    done = json.loads((run / "completed.json").read_text())
    m = done["best_validation"]
    rows.append({"model": variant, "mode": MODE, "best_epoch": done["best_epoch"],
                 "val_accuracy": m["accuracy"], "val_balanced_accuracy": m["balanced_accuracy"], "val_macro_f1": m["macro_f1"]})
    history = [json.loads(line) for line in (run / "history.jsonl").read_text().splitlines()]
    axes[0].plot([h["epoch"] for h in history], [h["total_loss"] for h in history], label=variant)
    axes[1].plot([h["epoch"] for h in history], [h["validation"]["balanced_accuracy"] for h in history], label=variant)
for ax, title in zip(axes, ["Training objective", "Validation balanced accuracy"]):
    ax.set(xlabel="Epoch", title=title)
    ax.legend()
fig.tight_layout()
plt.show()
display(pd.DataFrame(rows))

## 7. 完整训练后的一次最终评价（默认关闭）

确认预先选择的方法全部完成，并保持配置固定，再把设置格的
`RUN_FINAL_EVALUATION` 设为 `True`（或在下面一格上方赋值），执行这一格。

评价包含 OCT2017 test、NEH、OCT-C8。NEH 去掉 DME logit 后重新 softmax，
另存四类原始概率；OCT-C8 仅使用共同四类。全部结果都应保留。
测试完成后再修改模型属于新的探索，不再是未接触测试集的确认性比较。

In [ ]:
if not RUN_FINAL_EVALUATION:
    print("Final evaluation is off. No test predictions produced.")
elif MODE != "full":
    raise ValueError("Smoke checkpoints cannot be evaluated as final models.")
else:
    subprocess.run([sys.executable, "-u", "src/evaluate.py", "--root", str(RUN_ROOT),
                    "--config", str(CONFIG), "--experiments", str(EXPERIMENTS), "--archive", str(MIRROR)], check=True)
    subprocess.run([sys.executable, "src/aggregate.py", "--root", str(RUN_ROOT)], check=True)
    for name in ("summary.json", "combined_results.csv", "matched_contrasts.json", "RESULTS.md", "frozen_before_test.json"):
        shutil.copy2(RUN_ROOT / name, MIRROR / name)
    display(pd.read_csv(RUN_ROOT / "combined_results.csv"))
    print("Saved all outputs to:", MIRROR)

## 解释与记录

- 单次运行没有可估计的 seed 标准差。bootstrap 区间只描述该次运行的抽样不确定性。
- OCT-C8 的 CNV/DME 样本很少；看 macro-F1、每类 recall 和样本数，不只看 accuracy。
- 模型和论文未说明的细节见 `docs/METHODS.md`；本实现不保证达到论文数值。
- GPU 不足时可减小 batch，但应对所有比较方法一致修改，使用新 RUN_NAME 并记录。
- 在此填写你**实际观察到**的 validation / test 发现；不要把 smoke 输出作为正式结果。

Reference: Chutani & Yalavarthy (2026), DOI 10.1117/1.JBO.31.8.086006.